<a href="https://www.kaggle.com/code/kri500/ternery-nn-1-00?scriptVersionId=349508223" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

BATCH_SIZE = 256
EPOCHS = 10
LR = 1e-3

Using: cpu


In [3]:
class TernarizeSTE(torch.autograd.Function):
    """
    Forward: ternarize weights to {-1, 0, +1} * alpha (Ternary Weight Network, Li et al. 2016)
    Backward: straight-through estimator (pretend quantization was the identity function,
    so gradients flow to the latent full-precision weights unchanged).
    """
    @staticmethod
    def forward(ctx, w):
        delta = 0.7 * w.abs().mean()          # threshold, scales with weight magnitude
        mask = (w.abs() > delta).float()
        ternary = torch.sign(w) * mask         # values in {-1, 0, 1}
        num_nonzero = mask.sum().clamp(min=1)
        alpha = (w.abs() * mask).sum() / num_nonzero   # scaling factor
        return ternary * alpha

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output   # straight-through: identity gradient


class TernaryLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)

    def forward(self, x):
        w_t = TernarizeSTE.apply(self.weight)
        return F.linear(x, w_t, self.bias)


class TernaryConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, padding=0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_ch, in_ch, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_ch))
        self.padding = padding
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)

    def forward(self, x):
        w_t = TernarizeSTE.apply(self.weight)
        return F.conv2d(x, w_t, self.bias, padding=self.padding)

In [4]:
class TernaryCharNet(nn.Module):
    def __init__(self, num_classes=47):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)        # full precision
        self.conv2 = TernaryConv2d(32, 64, 3, padding=1)    # ternary
        self.pool = nn.MaxPool2d(2)
        self.fc1 = TernaryLinear(64 * 7 * 7, 128)           # ternary
        self.fc2 = nn.Linear(128, num_classes)              # full precision
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [5]:
# EMNIST images are stored rotated/flipped relative to how you'd view them
fix_orientation = transforms.Lambda(
    lambda img: img.rotate(-90).transpose(Image.FLIP_LEFT_RIGHT)
)

transform = transforms.Compose([
    fix_orientation,
    transforms.ToTensor(),
    transforms.Normalize((0.1751,), (0.3332,)),  # EMNIST mean/std
])

# NOTE: On Kaggle you need Settings -> Internet -> On for this download to work.
train_set = torchvision.datasets.EMNIST(root="./data", split="balanced",
                                         train=True, download=True, transform=transform)
test_set = torchvision.datasets.EMNIST(root="./data", split="balanced",
                                        train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"{len(train_set)} train / {len(test_set)} test, {len(train_set.classes)} classes")

100%|██████████| 562M/562M [00:06<00:00, 85.2MB/s]


112800 train / 18800 test, 47 classes


In [6]:
model = TernaryCharNet(num_classes=len(train_set.classes)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_set)
    test_acc = evaluate(test_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {train_loss:.4f} | test acc {test_acc:.4f}")

Epoch 1/10 | loss 1.1735 | test acc 0.8215
Epoch 2/10 | loss 0.6129 | test acc 0.8416
Epoch 3/10 | loss 0.5396 | test acc 0.8563
Epoch 4/10 | loss 0.4958 | test acc 0.8584
Epoch 5/10 | loss 0.4638 | test acc 0.8641
Epoch 6/10 | loss 0.4426 | test acc 0.8650
Epoch 7/10 | loss 0.4230 | test acc 0.8679
Epoch 8/10 | loss 0.4081 | test acc 0.8693
Epoch 9/10 | loss 0.3944 | test acc 0.8721
Epoch 10/10 | loss 0.3842 | test acc 0.8713


In [7]:
w = model.conv2.weight.detach()
w_t = TernarizeSTE.apply(w)
unique_vals = torch.unique(w_t / (w_t.abs().max() + 1e-9))
print("Distinct scaled weight levels (should collapse to ~3 groups):", unique_vals.numel())
print("Fraction of pruned (zero) weights:", (w_t == 0).float().mean().item())

Distinct scaled weight levels (should collapse to ~3 groups): 3
Fraction of pruned (zero) weights: 0.4811197817325592
